# 测试

来源:
- https://docs.langchain.com/mcp/test/index
- https://docs.langchain.com/mcp/test/unit-testing
- https://docs.langchain.com/mcp/test/integration-testing
- https://docs.langchain.com/mcp/test/evals

本笔记覆盖LangGraph应用测试的三个层次：单元测试、集成测试和评估(Evals)。

---
## 1. 测试概述

LangGraph测试策略分为三个层次：

| 层次 | 目标 | 范围 | 运行频率 |
|------|------|------|----------|
| **单元测试** | 验证单个节点/组件逻辑 | 无外部依赖 | 每次提交 |
| **集成测试** | 验证图结构和Agent交互 | 含外部依赖(LLM/DB) | 每日/PR |
| **评估(Evals)** | 验证整体质量和行为 | 完整系统 | 发布前 |

### 测试金字塔

```
        /\
       /  \     评估 (少量, 高成本)
      /    \
     /------\
    / 集成   \   集成测试 (中等量)
   /----------\
  /   单元     \  单元测试 (大量, 快速)
 /______________\
```

---
## 2. 单元测试

单元测试聚焦于独立的节点函数，不依赖LLM调用或外部服务。

### 测试纯函数节点

```python
import pytest
from typing import TypedDict

class TestState(TypedDict):
    value: int
    result: str

# 待测试的节点函数
def double_value(state: TestState) -> TestState:
    return {"value": state["value"] * 2}

def format_result(state: TestState) -> TestState:
    return {"result": f"最终值: {state['value']}"}


# 单元测试
class TestPureFunctions:
    def test_double_value(self):
        result = double_value({"value": 5, "result": ""})
        assert result["value"] == 10
    
    def test_double_value_zero(self):
        result = double_value({"value": 0, "result": ""})
        assert result["value"] == 0
    
    def test_double_value_negative(self):
        result = double_value({"value": -3, "result": ""})
        assert result["value"] == -6
    
    def test_format_result(self):
        result = format_result({"value": 10, "result": ""})
        assert result["result"] == "最终值: 10"
```

### 测试图编译

```python
from langgraph.graph import StateGraph, START, END

class TestGraphStructure:
    """测试图结构是否正确"""
    
    def test_graph_compiles(self):
        builder = StateGraph(TestState)
        builder.add_node("double", double_value)
        builder.add_node("format", format_result)
        builder.add_edge(START, "double")
        builder.add_edge("double", "format")
        builder.add_edge("format", END)
        
        graph = builder.compile()
        assert graph is not None
        result = graph.invoke({"value": 4, "result": ""})
        assert result["value"] == 8
        assert result["result"] == "最终值: 8"
    
    def test_graph_has_correct_nodes(self):
        builder = StateGraph(TestState)
        builder.add_node("node_a", lambda s: s)
        builder.add_node("node_b", lambda s: s)
        graph = builder.compile()
        
        # 检查节点是否存在
        assert "node_a" in graph.nodes
        assert "node_b" in graph.nodes
```

### Mock LLM调用

```python
from unittest.mock import MagicMock, patch

class TestWithMockLLM:
    """使用mock模拟LLM调用"""
    
    def test_router_logic(self):
        """测试路由逻辑无需真实LLM"""
        mock_llm = MagicMock()
        mock_llm.with_structured_output.return_value.invoke.return_value = \
            MagicMock(route="billing")
        
        state = {"messages": [{"role": "human", "content": "我的账单有问题"}]}
        
        # 注入mock并测试
        with patch("my_module.llm", mock_llm):
            from my_module import router_agent
            result = router_agent(state)
            assert result.goto == "agent_billing"
    
    def test_tool_invocation(self):
        """测试工具调用逻辑"""
        mock_tool = MagicMock()
        mock_tool.return_value = "模拟结果"
        
        result = mock_tool("test_input")
        mock_tool.assert_called_once_with("test_input")
        assert result == "模拟结果"
```

---
## 3. 集成测试

集成测试验证多个组件的交互，通常使用真实LLM但控制测试范围。

### 测试完整图执行

```python
import pytest

class TestIntegration:
    """集成测试 - 使用真实LLM但限制范围"""
    
    @pytest.mark.integration
    def test_full_graph_execution(self):
        """测试完整图执行流程"""
        graph = build_my_graph()
        result = graph.invoke({
            "messages": [("human", "Hello, 你是谁？")]
        })
        
        assert "messages" in result
        assert len(result["messages"]) >= 1
        assert result["messages"][-1].content is not None
    
    @pytest.mark.integration
    def test_handoff_mechanism(self):
        """测试交接机制"""
        graph = build_customer_support_graph()
        result = graph.invoke({
            "messages": [("human", "我被多收了费用，需要帮助")]
        })
        
        # 验证最终回复包含账单相关信息
        final_msg = result["messages"][-1].content.lower()
        assert any(word in final_msg for word in ["账单", "费用", "退款", "charge", "bill"])
    
    @pytest.mark.integration
    def test_tool_calling(self):
        """测试工具调用"""
        graph = build_tool_using_graph()
        result = graph.invoke({
            "messages": [("human", "搜索关于AI的最新新闻")]
        })
        
        # 验证搜索工具被调用
        messages = result["messages"]
        tool_calls = [m for m in messages if m.type == "tool" or hasattr(m, 'tool_calls')]
        assert len(tool_calls) > 0
```

### 测试检查点和中断

```python
class TestCheckpointing:
    """测试检查点和中断恢复"""
    
    @pytest.mark.integration
    def test_checkpoint_persistence(self):
        from langgraph.checkpoint import MemorySaver
        
        checkpointer = MemorySaver()
        graph = build_graph_with_interrupt()
        graph = graph.compile(checkpointer=checkpointer)
        
        # 首次执行（会在中断点暂停）
        config = {"configurable": {"thread_id": "test-1"}}
        result = graph.invoke(
            {"messages": [("human", "执行任务")]},
            config
        )
        
        # 验证中断发生
        assert result.get("interrupted") is True
        
        # 恢复执行
        result = graph.invoke(
            Command(resume={"approved": True}),
            config
        )
        assert "messages" in result
```

---
## 4. 评估 (Evals)

评估用于衡量Agent系统的整体质量和行为是否符合预期。

### 评估维度

| 维度 | 度量方法 | 目标 |
|------|----------|------|
| **正确性** | 输出与标准答案对比 | 准确率 > 90% |
| **相关性** | 回答与问题的相关度 | 相关性评分 > 4/5 |
| **安全性** | 检查是否产生有害输出 | 零有害输出 |
| **效率** | 执行时间、Token消耗 | 时间 < 10s, Token < 2000 |
| **鲁棒性** | 对异常输入的响应 | 优雅降级而非崩溃 |

### 代码示例：评估框架

```python
from typing import List, Dict, Any
from dataclasses import dataclass

@dataclass
class EvalCase:
    """评估测试用例"""
    input: str
    expected_output: str
    expected_route: str = None
    expected_tools: List[str] = None

@dataclass
class EvalResult:
    """评估结果"""
    case: EvalCase
    passed: bool
    actual_output: str
    score: float
    details: str


class LangGraphEval:
    """LangGraph评估框架"""
    
    def __init__(self, graph, eval_cases: List[EvalCase]):
        self.graph = graph
        self.cases = eval_cases
    
    def run(self) -> List[EvalResult]:
        results = []
        for case in self.cases:
            try:
                result = self.graph.invoke({
                    "messages": [("human", case.input)]
                })
                actual = result["messages"][-1].content
                
                # LLM作为评判者
                score = self._llm_judge(case.expected_output, actual)
                passed = score >= 0.7
                
                results.append(EvalResult(
                    case=case, passed=passed,
                    actual_output=actual, score=score,
                    details=""
                ))
            except Exception as e:
                results.append(EvalResult(
                    case=case, passed=False,
                    actual_output="", score=0.0,
                    details=str(e)
                ))
        return results
    
    def _llm_judge(self, expected: str, actual: str) -> float:
        """使用LLM评估回答质量"""
        judge_prompt = f"""
        预期答案: {expected}
        实际回答: {actual}
        请给出0-1之间的相似度评分:
        """
        response = judge_llm.invoke(judge_prompt)
        return extract_score(response.content)
    
    def report(self, results: List[EvalResult]):
        passed = sum(1 for r in results if r.passed)
        total = len(results)
        print(f"通过率: {passed}/{total} ({passed/total*100:.1f}%)")
        print(f"平均分: {sum(r.score for r in results)/total:.3f}")
        
        for r in results:
            status = "✓" if r.passed else "✗"
            print(f"{status} [{r.score:.2f}] {r.case.input[:50]}...")
            if not r.passed:
                print(f"   预期: {r.case.expected_output[:100]}...")
                print(f"   实际: {r.actual_output[:100]}...")


# 使用示例
eval_cases = [
    EvalCase("账单查询", "请提供您的账户信息以便查询账单", expected_route="billing"),
    EvalCase("技术故障", "请描述具体的技术问题", expected_route="tech"),
    EvalCase("你好", "你好！有什么可以帮助你的？", expected_route="general"),
]

evaluator = LangGraphEval(support_graph, eval_cases)
results = evaluator.run()
evaluator.report(results)
```

### 持续评估流水线

```python
# CI/CD 集成示例 (pytest)
import json

def test_regression_suite():
    """回归测试套件 - 每次发布前运行"""
    with open("eval_cases.json") as f:
        cases = [EvalCase(**c) for c in json.load(f)]
    
    evaluator = LangGraphEval(load_production_graph(), cases)
    results = evaluator.run()
    
    pass_rate = sum(1 for r in results if r.passed) / len(results)
    assert pass_rate >= 0.85, f"回归通过率 {pass_rate:.1%} 低于阈值 85%"


def test_edge_cases():
    """边界情况测试"""
    edge_cases = [
        EvalCase("", "请输入您的问题"),  # 空输入
        EvalCase("!@#$%^&*()", "请描述您的问题"),  # 特殊字符
        EvalCase("a" * 10000, "您的问题太长"),  # 超长输入
    ]
    evaluator = LangGraphEval(load_graph(), edge_cases)
    results = evaluator.run()
    
    for r in results:
        assert r.passed, f"边界情况失败: {r.case.input}"
```

---
## 5. 测试最佳实践

1. **分层测试**: 从单元测试到Eval逐层验证
2. **Mock外部依赖**: 测试核心逻辑时不依赖LLM
3. **快照测试**: 保存典型输出作为基准
4. **回归测试**: 每次修改后运行完整测试套件
5. **监控集成**: 将Evals集成到LangSmith监控
6. **测试数据管理**: 使用fixture管理测试数据

### 推荐的pytest配置

```python
# pytest.ini
[pytest]
markers =
    unit: 单元测试 (快速)
    integration: 集成测试 (需要LLM)
    eval: 评估测试 (慢)
    regression: 回归测试
testpaths = tests
```

```bash
# 运行不同层次的测试
pytest -m unit          # 仅单元测试
pytest -m integration   # 仅集成测试
pytest -m eval          # 仅评估
pytest -m "not eval"    # 除评估外所有测试
```

## 6. 参考链接

- [测试概述](https://docs.langchain.com/mcp/test/index)
- [单元测试指南](https://docs.langchain.com/mcp/test/unit-testing)
- [集成测试指南](https://docs.langchain.com/mcp/test/integration-testing)
- [评估指南](https://docs.langchain.com/mcp/test/evals)
- [LangSmith 评估](https://docs.smith.langchain.com/evaluation)
- [pytest 文档](https://docs.pytest.org/)